# Mamba 状态空间模型教程

本教程介绍 Mamba (Selective State Space Model) 的核心概念和实现。

## 目录
1. SSM 基础
2. 离散化
3. 选择性机制
4. Mamba 架构
5. 完整模型使用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

from mamba import (
    MambaConfig,
    SelectiveSSM,
    MambaBlock,
    MambaModel,
    MambaForCausalLM,
    create_mamba_model,
    count_parameters,
    discretize_ssm,
    selective_scan,
)

## 1. 状态空间模型 (SSM) 基础

连续时间 SSM:
$$h'(t) = A h(t) + B x(t)$$
$$y(t) = C h(t) + D x(t)$$

In [ ]:
# 简单 SSM 示例
def simple_ssm_step(h, x, A, B, C, D):
    """单步 SSM 计算"""
    h_new = A * h + B * x
    y = C * h_new + D * x
    return h_new, y

# 参数
A, B, C, D = 0.9, 0.1, 1.0, 0.0
h = 0.0

# 输入信号 (脉冲)
x_seq = np.zeros(50)
x_seq[10] = 1.0

# 运行 SSM
y_seq = []
h_seq = []
for x in x_seq:
    h, y = simple_ssm_step(h, x, A, B, C, D)
    y_seq.append(y)
    h_seq.append(h)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].stem(x_seq)
axes[0].set_title('Input x(t)')
axes[1].plot(h_seq)
axes[1].set_title('Hidden State h(t)')
axes[2].plot(y_seq)
axes[2].set_title('Output y(t)')
plt.tight_layout()
plt.show()

## 2. 离散化方法

将连续 SSM 转换为离散形式，常用 Zero-Order Hold (ZOH):
$$\bar{A} = \exp(\Delta A)$$
$$\bar{B} \approx \Delta B$$

In [ ]:
# 离散化示例
d_inner, d_state = 8, 4
batch_size, seq_len = 2, 10

A = np.random.randn(d_inner, d_state)
B = np.random.randn(batch_size, seq_len, d_state)
delta = np.abs(np.random.randn(batch_size, seq_len, d_inner)) * 0.1

# 比较不同离散化方法
A_zoh, B_zoh = discretize_ssm(A, B, delta, method='zoh')
A_euler, B_euler = discretize_ssm(A, B, delta, method='euler')
A_bilinear, B_bilinear = discretize_ssm(A, B, delta, method='bilinear')

print(f"ZOH A_bar shape: {A_zoh.shape}")
print(f"Euler A_bar shape: {A_euler.shape}")
print(f"Bilinear A_bar shape: {A_bilinear.shape}")

## 3. 选择性机制

Mamba 的核心创新：使 Δ, B, C 依赖于输入

In [ ]:
# 选择性扫描
batch_size, seq_len, d_inner, d_state = 2, 20, 16, 8

x = np.random.randn(batch_size, seq_len, d_inner)
delta = np.abs(np.random.randn(batch_size, seq_len, d_inner)) * 0.1
A = -np.abs(np.random.randn(d_inner, d_state))  # 负值保证稳定性
B = np.random.randn(batch_size, seq_len, d_state)
C = np.random.randn(batch_size, seq_len, d_state)
D = np.ones(d_inner)

y = selective_scan(x, delta, A, B, C, D)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

## 4. SelectiveSSM 模块

In [ ]:
# 创建 SelectiveSSM
ssm = SelectiveSSM(
    d_model=64,
    d_state=16,
    d_conv=4,
    expand=2,
    dt_rank=4,
)

x = np.random.randn(2, 10, 64)
y, _ = ssm(x)

print(f"Input: {x.shape}")
print(f"Output: {y.shape}")

## 5. 完整 Mamba 模型

In [ ]:
# 创建配置
config = MambaConfig(
    d_model=64,
    n_layers=4,
    d_state=16,
    vocab_size=1000,
)

print(f"d_model: {config.d_model}")
print(f"d_inner: {config.d_inner}")
print(f"n_layers: {config.n_layers}")
print(f"dt_rank: {config.dt_rank}")

In [ ]:
# 创建语言模型
model = MambaForCausalLM(config)

# 前向传播
input_ids = np.random.randint(0, 1000, size=(2, 10))
result = model(input_ids)

print(f"Logits shape: {result['logits'].shape}")

In [ ]:
# 计算损失
labels = np.random.randint(0, 1000, size=(2, 10))
result = model(input_ids, labels=labels)

print(f"Loss: {result['loss']:.4f}")

In [ ]:
# 文本生成
prompt = np.array([[1, 2, 3]])
generated = model.generate(
    prompt,
    max_new_tokens=10,
    temperature=0.8,
)

print(f"Prompt: {prompt[0]}")
print(f"Generated: {generated[0]}")

## 6. 参数量分析

In [ ]:
# 不同规模模型的参数量
sizes = ['small', 'base', 'large', 'xlarge']

for size in sizes:
    model = create_mamba_model(size)
    params = count_parameters(model.config)
    print(f"{size:8s}: {params['total_millions']:.1f}M parameters")

In [ ]:
# 参数分布
config = MambaConfig(d_model=768, n_layers=24, vocab_size=50264)
params = count_parameters(config)

labels = ['Embedding', 'All Layers', 'Final Norm']
values = [params['embedding'], params['all_layers'], params['final_norm']]

plt.figure(figsize=(8, 5))
plt.pie(values, labels=labels, autopct='%1.1f%%')
plt.title(f'Mamba-130M Parameter Distribution\nTotal: {params["total_millions"]:.1f}M')
plt.show()

## 7. 复杂度对比

Mamba vs Transformer 的复杂度对比

In [ ]:
# 复杂度可视化
seq_lengths = np.arange(100, 10001, 100)
d_model = 768
d_state = 16

# Transformer: O(L^2 * D)
transformer_flops = seq_lengths ** 2 * d_model

# Mamba: O(L * D * N)
mamba_flops = seq_lengths * d_model * d_state

plt.figure(figsize=(10, 5))
plt.plot(seq_lengths, transformer_flops / 1e9, label='Transformer O(L²D)', linewidth=2)
plt.plot(seq_lengths, mamba_flops / 1e9, label='Mamba O(LDN)', linewidth=2)
plt.xlabel('Sequence Length')
plt.ylabel('FLOPs (Billions)')
plt.title('Complexity Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 总结

Mamba 的核心优势:
1. **线性复杂度**: O(L) vs Transformer 的 O(L²)
2. **固定状态大小**: 推理时内存不随序列长度增长
3. **选择性机制**: 实现内容感知的信息过滤